# 02 · Retrieval pipeline
Dense (Chroma cosine) + sparse (BM25) → Reciprocal Rank Fusion → rerank, always inside an **authorization filter**.
Requires notebook 01 to have been run (it reads the same store).

In [1]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env  (set NB_OFFLINE=0 or edit below)
OFFLINE = os.getenv("NB_OFFLINE", "1") == "1"
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

mode            : OFFLINE (mock)
vector store    : local (/home/claude/proj/v1/rag_service/chroma_notebook)
embedding       : hash512
LLM providers   : ['gemini', 'groq', 'openai']


In [2]:
from app.retriever import HybridRetriever, build_where
from app.store import get_store
assert get_store().count() > 0, "Run notebook 01 first"
def show(q, subj, k=3):
    r = HybridRetriever(where=build_where([subj]), top_k=k)
    print(f"Q: {q}   [scope={subj}]")
    for d, s in r.search(q): print(f"  {s:.3f} | {d.metadata['file_name']:<22} p.{d.metadata['page']} | {d.page_content[:70]!r}")
    print("  trace:", r.last_trace, "\n")
show("Which organelle produces ATP?", "BIO101")
show("When does total internal reflection occur?", "PHY101")

Q: Which organelle produces ATP?   [scope=BIO101]
  0.738 | bio_notes.pdf          p.1 | 'Cell Biology Notes\nPhotosynthesis occurs in the chloroplasts of plant '
  0.731 | bio_slides.pptx        p.1 | 'Cell Biology Notes\nPhotosynthesis occurs in the chloroplasts of plant '
  0.218 | bio_notes.pdf          p.2 | 'Genetics\nDNA replication is semi-conservative: each new double helix c'
  trace: {'dense_hits': 3, 'sparse_hits': 2, 'fused_candidates': 3, 'returned': 3, 'top_score': 0.738} 

Q: When does total internal reflection occur?   [scope=PHY101]
  0.757 | physics.docx           p.2 | "Optics\nSnell's law relates the angles of incidence and refraction: n1 "
  0.737 | physics_facts.xlsx     p.1 | "Optics\nSheet Optics:\nTopic | Fact\nOptics | Snell's law relates the ang"
  0.218 | physics.docx           p.1 | "Mechanics\nNewton's second law states that the net force on an object e"
  trace: {'dense_hits': 3, 'sparse_hits': 2, 'fused_candidates': 3, 'returned': 3, 'top_score': 0.7575

## LangChain retriever interface
`HybridRetriever` is a standard `BaseRetriever`, so it also works in any LangChain chain via `.invoke()`.

In [3]:
r = HybridRetriever(where=build_where(["CS201"]), top_k=2)
for d in r.invoke("What is third normal form?"): print(d.metadata["file_name"], "|", d.page_content[:80])

db_page.html | DB
Database Normalization
First normal form requires that every column holds ato
db_facts.csv | CSV
topic | fact
Database Normalization | First normal form requires that every 


## Authorization isolation: a Physics student must never see Biology chunks

In [4]:
r = HybridRetriever(where=build_where(["PHY101"]), top_k=10)
hits = r.search("Which organelle produces ATP?")
print({d.metadata["subject_code"] for d, _ in hits})
assert all(d.metadata["subject_code"] == "PHY101" for d, _ in hits)

{'PHY101'}


## Retrieval quality on the synthetic QA set (hit@3)

In [5]:
import synthetic, pandas as pd
rows = []
for subj, q, kw, _ in synthetic.QA:
    top = HybridRetriever(where=build_where([subj]), top_k=3).search(q)
    rows.append({"subject": subj, "question": q, "hit@3": any(kw in d.page_content.lower() for d, _ in top)})
df = pd.DataFrame(rows); print("hit@3 =", df["hit@3"].mean()); df

hit@3 = 1.0


,subject,question,hit@3
0,BIO101,Where does photosynthesis take place in a plan...,True
1,BIO101,Which organelle produces ATP?,True
2,BIO101,What does semi-conservative DNA replication mean?,True
3,BIO101,Which enzyme adds nucleotides during DNA repli...,True
4,PHY101,State Newton's second law.,True
5,PHY101,What is the formula for kinetic energy?,True
6,PHY101,When does total internal reflection occur?,True
7,CS201,What does ACID stand for in databases?,True
8,CS201,What is third normal form?,True
9,CS201,How are deadlocks resolved?,True
